## Hw 1b: Experimental Analysis of a Shallow Neural Network

Import all necessary libraries

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from training_utils import get_model, train_model, valid_metrics, apply_augmentations, show_batch

### 1. Validation set
First 10,000 images of the training set for validation, the remaining 50,000 for training

In [ ]:
train_df = pd.read_csv("fashion_mnist/fashion-mnist_train.csv")

labels_np = train_df.iloc[:, 0].astype(np.int64).values.copy()
images_np = train_df.iloc[:, 1:].astype(np.float32).values.copy()

X = torch.tensor(images_np, dtype=torch.float32) / 255.0
Y = torch.tensor(labels_np, dtype=torch.long)

X_valid, Y_valid = X[:10000], Y[:10000]
X_train, Y_train = X[10000:], Y[10000:]

X_train.shape, X_valid.shape

### 2. Learning rate selection
Hidden size 50, 500 epochs, new model for every learning rate. Same seed before each model so they all start from the same weights

In [ ]:
epochs = 500
learning_rates = [1, 0.1, 0.01, 0.001, 0.0001, 0.00001]

lr_results = []
for lr in learning_rates:
    torch.manual_seed(0)
    model = get_model(hidden_size=50)
    train_loss = train_model(model, X_train, Y_train, learning_rate=lr, epochs=epochs)
    valid_loss, valid_acc = valid_metrics(model, X_valid, Y_valid)
    lr_results.append({"learning_rate": lr, "train_loss": train_loss,
                       "valid_loss": valid_loss, "valid_acc": valid_acc})

lr_df = pd.DataFrame(lr_results)
lr_df

Interpolate between the two best learning rates. We rank them by validation loss instead of accuracy, it is smoother and also shows when a model starts overfitting

In [ ]:
low = lr_df.nsmallest(2, "valid_loss")["learning_rate"].min()
new_lrs = [low * m for m in [2, 3, 5, 7]]
print("new learning rates:", new_lrs)

new_results = []
for lr in new_lrs:
    torch.manual_seed(0)
    model = get_model(hidden_size=50)
    train_loss = train_model(model, X_train, Y_train, learning_rate=lr, epochs=epochs)
    valid_loss, valid_acc = valid_metrics(model, X_valid, Y_valid)
    new_results.append({"learning_rate": lr, "train_loss": train_loss,
                        "valid_loss": valid_loss, "valid_acc": valid_acc})

lr_all_df = pd.concat([lr_df, pd.DataFrame(new_results)]).sort_values("learning_rate").reset_index(drop=True)
best_lr = lr_all_df.loc[lr_all_df["valid_loss"].idxmin(), "learning_rate"]
print("best learning rate:", best_lr)
lr_all_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
lr_all_df.plot(x="learning_rate", y=["train_loss", "valid_loss"], logx=True, logy=True, marker="o", ax=ax[0], title="Loss")
lr_all_df.plot(x="learning_rate", y="valid_acc", logx=True, marker="o", ax=ax[1], title="Validation accuracy", legend=False)
plt.show()

### 3. Effect of hidden layer size
Learning rate fixed to the best one from above

In [ ]:
hidden_sizes = [10, 50, 100, 300, 1000, 2000]

hidden_results = []
for h in hidden_sizes:
    torch.manual_seed(0)
    model = get_model(hidden_size=h)
    train_loss = train_model(model, X_train, Y_train, learning_rate=best_lr, epochs=epochs)
    valid_loss, valid_acc = valid_metrics(model, X_valid, Y_valid)
    hidden_results.append({"hidden_size": h, "train_loss": train_loss,
                           "valid_loss": valid_loss, "valid_acc": valid_acc})

hidden_df = pd.DataFrame(hidden_results)
best_hidden = int(hidden_df.loc[hidden_df["valid_loss"].idxmin(), "hidden_size"])
print("best hidden size:", best_hidden)
hidden_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hidden_df.plot(x="hidden_size", y=["train_loss", "valid_loss"], logx=True, marker="o", ax=ax[0], title="Loss")
hidden_df.plot(x="hidden_size", y="valid_acc", logx=True, marker="o", ax=ax[1], title="Validation accuracy", legend=False)
plt.show()

### 4. Visualize transformed images
First row: original images. Rows 2 to 4: mild, moderate and aggressive

In [ ]:
imgs = X_train[:4]

# one call per image so every image gets its own random transform
rows = [imgs]
for level in ["mild", "moderate", "aggressive"]:
    rows.append(torch.cat([apply_augmentations(img, level=level) for img in imgs]))

show_batch(torch.cat(rows).reshape(-1, 28, 28))

### 5. Effect of data augmentation
Best learning rate and hidden size from above. Only the training images are augmented, so the train loss of the augmented runs is measured on augmented images

In [ ]:
aug_levels = [None, "mild", "moderate", "aggressive"]

aug_results = []
for level in aug_levels:
    torch.manual_seed(0)
    model = get_model(hidden_size=best_hidden)
    train_loss = train_model(model, X_train, Y_train, learning_rate=best_lr, epochs=epochs, level=level)
    valid_loss, valid_acc = valid_metrics(model, X_valid, Y_valid)
    aug_results.append({"augmentation": "baseline" if level is None else level, "train_loss": train_loss,
                        "valid_loss": valid_loss, "valid_acc": valid_acc})

aug_df = pd.DataFrame(aug_results)
aug_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
aug_df.plot.bar(x="augmentation", y=["train_loss", "valid_loss"], rot=0, ax=ax[0], title="Loss")
aug_df.plot.bar(x="augmentation", y="valid_acc", rot=0, ax=ax[1], title="Validation accuracy", legend=False)
plt.show()